modelo conceptual/lógico, decisiones (trazabilidad con unit_price), constraints, índices, consultas típicas, checks de calidad y evolución del esquema (tabla de devoluciones).

In [10]:
import sqlite3
import pandas as pd

conn=sqlite3.connect(":memory:")
def q(sql:str,params=None)->pd.DataFrame:
    return pd.read_sql_query(sql,conn,params=params or {})

# 1) Contexto del problema
# Queremos soportar: clientes, catálogo de productos, pedidos con líneas, pagos, devoluciones (extensible)
# Consultas típicas: ventas por periodo, top productos, AOV, cohortes de clientes, devoluciones, márgenes

# 2) Modelo conceptual (explicación resumida, orientada a entrevista)
# Entidades: Customer, Product, Order, OrderItem, Payment
# Relaciones:
# - Customer 1..N Order
# - Order 1..N OrderItem
# - Product 1..N OrderItem
# - Order 0..N Payment
# Decisiones:
# - Se separa OrderItem para resolver N..M entre Orders y Products y capturar qty + precio en el momento de compra
# - Se guarda unit_price en OrderItem para trazabilidad histórica (si el precio de Product cambia)

# 3) Modelo lógico (tablas y claves)
conn.executescript("""
DROP TABLE IF EXISTS customers;DROP TABLE IF EXISTS products;DROP TABLE IF EXISTS orders;DROP TABLE IF EXISTS order_items;DROP TABLE IF EXISTS payments;
CREATE TABLE customers(
  id_customer INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  region TEXT NOT NULL,
  signup_date TEXT NOT NULL
);
CREATE TABLE products(
  id_product INTEGER PRIMARY KEY,
  sku TEXT NOT NULL UNIQUE,
  category TEXT NOT NULL,
  price REAL NOT NULL CHECK(price>=0),
  active INTEGER NOT NULL CHECK(active IN (0,1))
);
CREATE TABLE orders(
  id_order INTEGER PRIMARY KEY,
  id_customer INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  status TEXT NOT NULL CHECK(status IN ('CREATED','PAID','SHIPPED','CANCELLED')),
  FOREIGN KEY(id_customer) REFERENCES customers(id_customer)
);
CREATE TABLE order_items(
  id_order INTEGER NOT NULL,
  id_product INTEGER NOT NULL,
  qty INTEGER NOT NULL CHECK(qty>0),
  unit_price REAL NOT NULL CHECK(unit_price>=0),
  PRIMARY KEY(id_order,id_product),
  FOREIGN KEY(id_order) REFERENCES orders(id_order),
  FOREIGN KEY(id_product) REFERENCES products(id_product)
);
CREATE TABLE payments(
  id_payment INTEGER PRIMARY KEY,
  id_order INTEGER NOT NULL,
  payment_date TEXT NOT NULL,
  amount REAL NOT NULL CHECK(amount>=0),
  method TEXT NOT NULL,
  FOREIGN KEY(id_order) REFERENCES orders(id_order)
);
""")

In [11]:
# Poblar mímimo
conn.executescript("""
INSERT INTO customers VALUES
(1,'Alice','Catalunya','2023-01-10'),
(2,'Bob','Madrid','2023-02-05');

INSERT INTO products VALUES
(1,'SKU-1','Electrónica',100.0,1),
(2,'SKU-2','Hogar',40.0,1);

INSERT INTO orders VALUES
(1,1,'2024-01-15','PAID'),
(2,2,'2024-02-20','SHIPPED');

INSERT INTO order_items VALUES
(1,1,1,100.0),
(1,2,2,40.0),
(2,2,1,40.0);

INSERT INTO payments VALUES
(1,1,'2024-01-16',180.0,'Card'),
(2,2,'2024-02-21',40.0,'Bizum');
""")
conn.commit()


In [12]:
# 4) Índices (decisión y justificación)
# Índices típicos: columnas de JOIN y filtros frecuentes por fecha/cliente
conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(id_customer)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_date ON orders(order_date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_payments_order ON payments(id_order)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_payments_date ON payments(payment_date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(id_order)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_items_product ON order_items(id_product)")
conn.commit()

In [13]:
# 5) Cómo demostrar integridad y esquema
print(q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"))
print(q("PRAGMA table_info(orders);"))

          name
0    customers
1  order_items
2       orders
3     payments
4     products
   cid         name     type  notnull dflt_value  pk
0    0     id_order  INTEGER        0       None   1
1    1  id_customer  INTEGER        1       None   0
2    2   order_date     TEXT        1       None   0
3    3       status     TEXT        1       None   0


In [14]:
# 6) Consultas típicas (cómo el modelo habilita negocio)
# 6.1 Ventas (revenue) por mes
print(q("""SELECT substr(payment_date,1,7) AS yyyy_mm,ROUND(SUM(amount),2) AS revenue FROM payments GROUP BY substr(payment_date,1,7) ORDER BY yyyy_mm;"""))

   yyyy_mm  revenue
0  2024-01    180.0
1  2024-02     40.0


In [15]:
# 6.2 Top productos por unidades vendidas (sobre pedidos pagados/enviados)
print(q("""SELECT p.category,oi.id_product,SUM(oi.qty) AS units FROM order_items oi JOIN orders o ON o.id_order=oi.id_order JOIN products p ON p.id_product=oi.id_product WHERE o.status IN ('PAID','SHIPPED') GROUP BY p.category,oi.id_product ORDER BY units DESC LIMIT 10;"""))

      category  id_product  units
0        Hogar           2      3
1  Electrónica           1      1


In [16]:
# 6.3 AOV (Average Order Value)
print(q("SELECT ROUND(AVG(amount),2) AS avg_order_value FROM payments;"))

   avg_order_value
0            110.0


In [17]:
# 7) Calidad de datos: checks útiles
print(q("SELECT COUNT(*) AS total,COUNT(DISTINCT id_customer) AS distinct_customers FROM customers;"))
print(q("SELECT SUM(CASE WHEN price<0 OR price IS NULL THEN 1 ELSE 0 END) AS bad_prices,COUNT(*) AS total FROM products;"))

   total  distinct_customers
0      2                   2
   bad_prices  total
0           0      2


In [18]:
# 8) Evolución del modelo (ejemplo): añadir devoluciones
# Decisión: tabla returns referenciando order_items (porque una devolución es por línea o por item)
conn.executescript("""
DROP TABLE IF EXISTS returns;
CREATE TABLE returns(
  id_return INTEGER PRIMARY KEY,
  id_order INTEGER NOT NULL,
  id_product INTEGER NOT NULL,
  return_date TEXT NOT NULL,
  qty_returned INTEGER NOT NULL CHECK(qty_returned>0),
  reason TEXT,
  FOREIGN KEY(id_order,id_product) REFERENCES order_items(id_order,id_product)
);
CREATE INDEX IF NOT EXISTS idx_returns_date ON returns(return_date);
""")
conn.commit()
print(q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"))

          name
0    customers
1  order_items
2       orders
3     payments
4     products
5      returns
